<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="../images/btp-banner.gif" alt="BTP A&C">
</div>

# BTP AI Workshop Day 2
## Grounding with SAP Gen AI Hub + Optional Advanced ReAct Agent

### The Use Case: Supplier Performance Analysis for QBR

At BestRun Technologies, the procurement team sources critical components for its smart-IoT security devices from a select group of trusted partners. Ahead of quarterly business-review (QBR) meetings, executives often ask questions such as:

- *"How is Techtronix Components Ltd. performing?"*
- *"What contractual penalties apply if PulseWave Materials Inc. misses a delivery?"*

The answers lie scattered across purchase orders, on-time delivery reports, quality-incident logs, supplier audit scores, and PDF contracts. Manually compiling this information from BestRun Technologies' SharePoint folders can take hours—still risking missed details, such as a late-delivery penalty clause or a recurring pattern of quality incidents with Sigma Electronics Co.

By indexing these documents and leveraging SAP's Generative AI Hub with document grounding, BestRun Technologies can automatically retrieve relevant delivery metrics, quality notes, and contract terms for any supplier, then generate a concise, context-rich briefing for its QBR deck. **This is Part 1 of our workshop: deterministic grounded Q&A.**

But then the CPO asks a harder question:

> *"The Techtronix contract expires in 60 days. Should we renew, renegotiate, or exit?"*

This question cannot be answered with a single retrieval. It requires weighing delivery performance against quality incidents, checking penalty clauses, and synthesizing a recommendation. The next step — which tool to call, which evidence to gather — depends on what we learn along the way. **This is Part 2: when the scenario evolves beyond deterministic workflows, and an agentic solution becomes necessary.**

## Why This Workshop Structure

Use this decision sequence throughout the workshop:
1. Define the business decision and required evidence.
2. Choose the simplest architecture that solves it.
3. Add an agent only when deterministic grounding is not enough.

Part 1 covers deterministic QBR questions. Part 2 covers the renewal decision where next steps depend on intermediate evidence.

### Workshop Roadmap

| Part | Section | ~Time | Outcome |
| --- | --- | --- | --- |
| **1 - Essential** | Setup (Steps 1.1-1.3) | 15 min | SDK + config + AI Core connectivity |
| | Before and After (Steps 1.4-1.6) | 15 min | Ungrounded vs grounded contrast |
| | Build: Grounded Q&A (Steps 1.7-1.9) | 30 min | Deterministic grounded query pattern |
| | Decide: Agent or Not? (Step 1.10) | 10 min | Architecture decision criteria |
| | Checkpoint | 10 min | Reflection + exercise |
| **2 - Advanced** | Agent Tools (Steps 2.1a-2.1b) | 10 min | Business tools + presentation tool |
| | ReAct Loop (Steps 2.2-2.3) | 15 min | Controlled JSON ReAct runtime |
| | Visualize and Trace (Steps 2.4-2.6) | 15 min | Recommendation, trace, HITL |

### Run Mode
- Execute cells in order and capture outputs at each checkpoint.

## Business Context and Success Criteria

### How the Scenario Evolves

**Part 1 — Grounded Q&A (Deterministic)**
The procurement team needs quick, accurate answers for the QBR deck:
- *"What are the payment terms in the Techtronix contract?"*
- *"Summarize Techtronix delivery performance."*

These are **single-path questions**: one retrieval, one generation, done. SAP Document Grounding + OrchestrationService handles them cleanly.

**Part 2 — Supplier Renewal Decision (Non-Deterministic)**
Now the CPO escalates: *"Should we renew, renegotiate, or exit the Techtronix contract?"*

Why this requires an agent:
| Dimension | Grounded Q&A | Agent (ReAct) |
| --- | --- | --- |
| Evidence needed | Single domain | Multiple domains (delivery + quality + contract) |
| Next step known? | Yes, fixed | No — depends on intermediate findings |
| Synthesis | Direct answer | Weighted recommendation with trade-offs |

The agent must decide: *Do I have enough delivery data? Should I check quality incidents next? What about contract penalties?* This adaptive reasoning justifies agentic orchestration.

### Data Landscape

| Document | Type | Key Content | Used In |
| --- | --- | --- | --- |
| `techtronix_components_ltd._contract.pdf` | Contract | Payment terms, penalties, termination, governing law | Part 1 + Part 2 |
| `sigma_electronics_co._contract.pdf` | Contract | Competitor benchmark terms | Part 1 Exercise |
| `pulsewave_materials_inc._contract.pdf` | Contract | Competitor benchmark terms | Part 1 Exercise |
| `on_time_delivery_report.txt` | Operational | Delivery performance metrics, delays | Part 1 + Part 2 |
| `quality_incidents_with_contacts.txt` | Quality | Incident logs, severity, resolution | Part 1 + Part 2 |
| `supplier_audits.pdf` | Audit | Supplier quality audit scores | Part 2 |
| `po_with_contacts.txt` | PO data | Purchase order details, contacts | Context |

### What Good Looks Like
By the end of this notebook, participants should be able to:
- Produce grounded answers from enterprise data using SAP services.
- Explain when **not** to use an agent.
- Build one advanced ReAct flow for non-deterministic multi-source decisions.

### SAP BTP Services Used and Why

| BTP Service | Role in This Workshop | Why Not a Generic Alternative? |
| --- | --- | --- |
| **SAP AI Core** | Secure runtime for model deployments | Enterprise auth, resource-group isolation, BTP identity integration |
| **Gen AI Hub SDK** | Pro-code LLM orchestration | Consistent API, version-pinned SDK, SAP support channel |
| **Document Grounding** | Managed retrieval over indexed docs | No custom vector DB ops; managed chunking, embedding, indexing |
| **BTP Object Store (S3)** | Source repository for business documents | Integrated with grounding pipeline; governed access |

This is intentionally **SAP-native** to support production-ready BTP AI adoption.

## Part 1 (Essential): Grounding via SAP Gen AI Hub

### Architecture Focus
This section demonstrates **Grounding** as an enterprise retrieval pattern using SAP-managed services.

```text
Business Question
    -> SAP Document Grounding Retrieval API (Gen AI Hub)
    -> Relevant chunks from indexed documents (BTP Object Store pipeline)
    -> SAP OrchestrationService via Gen AI Hub SDK
    -> Grounded answer with evidence
```

### Why This Matters for BTP Customers
- Keeps security, access, and model execution within SAP AI Core boundaries.
- Uses managed retrieval capabilities instead of custom retrieval infrastructure.
- Provides a repeatable pro-code pattern for enterprise apps.

---
### 🔧 Setup & Configuration

## Learning Objectives and Requirements

### Learning Objectives
By the end of this hands-on, participants will be able to:
- Build grounded enterprise Q&A using SAP Document Grounding and SAP OrchestrationService.
- Distinguish deterministic workflows from non-deterministic agent workflows.
- Implement one advanced ReAct agent loop with guardrails and traceability.

### Requirements
- Shared pre-provisioned workshop environment
- Valid `.env` values for SAP AI Core and grounding repository
- Access to deployed orchestration URL and indexed data repository

### Data Scope for the Use Case
Documents in the repository include supplier contracts, on-time delivery data, quality incidents, and supplier audits.


### Step 1.1 - Install Required Packages

We use SAP's `generative-ai-hub-sdk` as the primary pro-code interface.

Why this SDK:
- Standard SAP-supported path for orchestration and model access.
- Consistent API patterns across workshop and production code.

If your environment is already provisioned, this cell is idempotent.
**Restart kernel** by clicking the "Restart" button after install if your platform requests it.


In [19]:
%pip install generative-ai-hub-sdk==4.12.4 --break-system-packages --quiet
%pip install python-dotenv requests --break-system-packages --quiet

print("Packages installed.")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Packages installed.


### Step 1.1b - Prerequisites Check

Run this cell to verify your environment is ready before proceeding. This checks:
- Python version compatibility
- Network connectivity to SAP AI Core
- Required packages are importable

If any check fails, see the **Troubleshooting** section after Step 1.3.

In [20]:
import sys
import socket

def check_prerequisites():
    """Validate environment before workshop starts."""
    checks_passed = 0
    checks_total = 3
    
    # Check 1: Python version
    py_version = sys.version_info
    if py_version >= (3, 9):
        print(f"[PASS] Python version: {py_version.major}.{py_version.minor}.{py_version.micro}")
        checks_passed += 1
    else:
        print(f"[FAIL] Python version {py_version.major}.{py_version.minor} - requires 3.9+")
    
    # Check 2: Required packages importable
    try:
        import requests
        from dotenv import load_dotenv
        from gen_ai_hub.orchestration.service import OrchestrationService
        print("[PASS] Required packages are importable")
        checks_passed += 1
    except ImportError as e:
        print(f"[FAIL] Missing package: {e.name}. Re-run the install cell above.")
    
    # Check 3: Network connectivity (basic DNS check)
    try:
        socket.setdefaulttimeout(5)
        socket.getaddrinfo("api.sap.com", 443)
        print("[PASS] Network connectivity OK")
        checks_passed += 1
    except socket.gaierror:
        print("[WARN] Cannot resolve api.sap.com - check network/VPN connection")
    
    print("-" * 50)
    if checks_passed == checks_total:
        print(f"All {checks_total} checks passed. Ready to proceed!")
    else:
        print(f"{checks_passed}/{checks_total} checks passed. Review issues above.")

check_prerequisites()

[PASS] Python version: 3.13.11
[PASS] Required packages are importable
[PASS] Network connectivity OK
--------------------------------------------------
All 3 checks passed. Ready to proceed!


### Step 1.2 - Imports and Global Constants

What this step does:
- Imports SAP Gen AI Hub SDK classes for orchestration.
- Imports support libraries for credentials and HTTP calls.
- Defines stable constants to keep workshop behavior deterministic.


In [21]:
import json
import os
import time
from typing import Any, Dict, List, Tuple

import requests
from dotenv import find_dotenv, load_dotenv

from gen_ai_hub.orchestration.service import OrchestrationService
from gen_ai_hub.orchestration.models.config import OrchestrationConfig
from gen_ai_hub.orchestration.models.template import Template
from gen_ai_hub.orchestration.models.llm import LLM
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage

# --- Global Constants ---
MODEL_NAME = "anthropic--claude-4.5-sonnet"
TOKEN_TIMEOUT_SECONDS = 30
HTTP_TIMEOUT_SECONDS = 45
TOKEN_REFRESH_BUFFER_SECONDS = 60  # Refresh token this many seconds before expiry

# --- Tool Name Constants (avoids magic strings) ---
TOOL_DELIVERY_PERFORMANCE = "query_delivery_performance"
TOOL_QUALITY_INCIDENTS = "query_quality_incidents"
TOOL_CONTRACT_TERMS = "query_contract_terms"

# --- Agent Response Types ---
ACTION_TYPE = "action"
FINAL_TYPE = "final"
RETRY_TYPE = "retry"

print("Imports and constants loaded.")

Imports and constants loaded.


### Step 1.3 - Load `.env` Configuration

We separate configuration from code using `.env` (required for clean enterprise practices).

Expected environment variable names:
- `AICORE_AUTH_URL`
- `AICORE_CLIENT_ID`
- `AICORE_CLIENT_SECRET`
- `AICORE_BASE_URL`
- `AICORE_RESOURCE_GROUP`
- `ORCH_DEPLOYMENT_URL`
- `DATA_REPOSITORY_ID`

SAP relevance:
- `ORCH_DEPLOYMENT_URL` points to your SAP AI Core orchestration deployment.
- `DATA_REPOSITORY_ID` points to your Document Grounding indexed repository.

> **📌 BTP Design Decision:** We use `OrchestrationService` from the Gen AI Hub SDK (not raw HTTP calls) because it provides a consistent, version-pinned API for LLM orchestration. This means your workshop code patterns transfer directly to production CAP applications.

In [22]:
dotenv_path = find_dotenv()
load_dotenv(dotenv_path=dotenv_path, override=True)

AICORE_AUTH_URL = os.getenv("AICORE_AUTH_URL")
AICORE_CLIENT_ID = os.getenv("AICORE_CLIENT_ID")
AICORE_CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET")
AICORE_BASE_URL = os.getenv("AICORE_BASE_URL")
AICORE_RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP")
ORCH_DEPLOYMENT_URL = os.getenv("ORCH_DEPLOYMENT_URL")
DATA_REPOSITORY_ID = os.getenv("DATA_REPOSITORY_ID")

required = {
    "AICORE_AUTH_URL": AICORE_AUTH_URL,
    "AICORE_CLIENT_ID": AICORE_CLIENT_ID,
    "AICORE_CLIENT_SECRET": AICORE_CLIENT_SECRET,
    "AICORE_BASE_URL": AICORE_BASE_URL,
    "AICORE_RESOURCE_GROUP": AICORE_RESOURCE_GROUP,
    "ORCH_DEPLOYMENT_URL": ORCH_DEPLOYMENT_URL,
    "DATA_REPOSITORY_ID": DATA_REPOSITORY_ID,
}
missing = [k for k, v in required.items() if not v]
if missing:
    raise EnvironmentError(f"Missing required .env variables: {missing}")

# Initialize orchestration service (available at this step)
orchestration_service = OrchestrationService(api_url=ORCH_DEPLOYMENT_URL)

# GroundingClient class is defined in Step 1.5. Initialize if already loaded,
# otherwise defer initialization so Step 1.3 can run without errors.
grounding_cls = globals().get("GroundingClient")
if grounding_cls is not None:
    grounding_client = grounding_cls(
        auth_url=AICORE_AUTH_URL,
        client_id=AICORE_CLIENT_ID,
        client_secret=AICORE_CLIENT_SECRET,
        base_url=AICORE_BASE_URL,
        resource_group=AICORE_RESOURCE_GROUP,
    )
    grounding_status = "initialized"
else:
    grounding_client = None
    grounding_status = "deferred until Step 1.5"

print(f"Loaded .env from: {dotenv_path}")
print(f"Resource group: {AICORE_RESOURCE_GROUP}")
print(f"Deployment ID: ...{ORCH_DEPLOYMENT_URL.split('/')[-1][-8:]}")  # Mask most of the ID
print(f"Data repository ID: ...{DATA_REPOSITORY_ID[-8:]}")  # Mask most of the ID
print(f"OrchestrationService initialized; GroundingClient {grounding_status}.")

Loaded .env from: /Users/I560298/Downloads/11.github_sync/grounding2603/files/.env
Resource group: user050-l2o-rg
Deployment ID: ...d37394fd
Data repository ID: ...603bc5ee
OrchestrationService initialized; GroundingClient initialized.


### Troubleshooting Common Issues

Use this table first when errors appear.

| Error | Likely Cause | Fix |
|-------|--------------|-----|
| `Missing required .env variables: [...]` | `.env` missing/incomplete | Ensure `.env` exists in project root with all required keys |
| `401 Unauthorized` on token request | Invalid client credentials | Verify `AICORE_CLIENT_ID` and `AICORE_CLIENT_SECRET` |
| `403 Forbidden` on API calls | Wrong resource group or missing permission | Verify `AICORE_RESOURCE_GROUP` |
| `Connection timeout` | Network/VPN issue | Check VPN and confirm `AICORE_BASE_URL` reachability |
| `No relevant grounded context was retrieved` | Query mismatch vs indexed docs | Rephrase query and verify `DATA_REPOSITORY_ID` |
| `JSONDecodeError` in agent | Malformed LLM JSON | Set `PARSE_DEBUG=True` and retry |
| `Unknown tool` error | Tool name mismatch | Agent retries automatically; verify prompt/tool names if repeated |

### Quick Decision Mini-Map
- Auth/config issue: re-run Steps 1.2 -> 1.3, then diagnostics.
- Empty grounding: retry Step 1.6 with clearer supplier/topic wording.
- Parse issue: set `PARSE_DEBUG=True`, re-run Steps 2.2 and 2.4.
- Dashboard issue: re-run Step 2.1b, then Step 2.4.

**Quick diagnostic commands:**

In [23]:
def run_diagnostics():
    """Run diagnostic checks to help troubleshoot common issues."""
    print("=" * 60)
    print("DIAGNOSTIC REPORT")
    print("=" * 60)
    
    # Check 1: Environment variables loaded
    print("\n1. Environment Variables:")
    env_vars = [
        "AICORE_AUTH_URL", "AICORE_CLIENT_ID", "AICORE_BASE_URL",
        "AICORE_RESOURCE_GROUP", "ORCH_DEPLOYMENT_URL", "DATA_REPOSITORY_ID"
    ]
    for var in env_vars:
        value = os.getenv(var)
        if value:
            # Mask sensitive values
            masked = f"...{value[-8:]}" if len(value) > 12 else "(set)"
            print(f"   [OK] {var}: {masked}")
        else:
            print(f"   [MISSING] {var}")
    
    # Check 2: Test token acquisition
    print("\n2. Token Acquisition:")
    try:
        if grounding_client:
            token = grounding_client._get_token()
            print(f"   [OK] Token acquired (length: {len(token)})")
        else:
            print("   [SKIP] GroundingClient not initialized")
    except Exception as e:
        print(f"   [FAIL] {type(e).__name__}: {e}")
    
    # Check 3: Test grounding API
    print("\n3. Grounding API:")
    try:
        _, sources, count = retrieve_grounding_context("test query", max_chunks=1)
        print(f"   [OK] API reachable (returned {count} chunk(s))")
    except Exception as e:
        print(f"   [FAIL] {type(e).__name__}: {e}")
    
    # Check 4: Test orchestration service
    print("\n4. Orchestration Service:")
    try:
        test_response = orchestration_service.run(
            config=OrchestrationConfig(
                template=Template(messages=[
                    SystemMessage("Reply with 'OK' only."),
                    UserMessage("Test")
                ]),
                llm=LLM(name=MODEL_NAME)
            )
        )
        content = test_response.orchestration_result.choices[0].message.content
        print(f"   [OK] Service responded: '{content[:50]}...'")
    except Exception as e:
        print(f"   [FAIL] {type(e).__name__}: {e}")
    
    print("\n" + "=" * 60)
    print("If all checks pass, your environment is ready!")
    print("=" * 60)

# Uncomment the line below to run diagnostics:
# run_diagnostics()

---
### Before vs. After: The Grounding Effect

### Step 1.4 - Quick Connection Check (No Grounding Yet)

Purpose:
- Validate that SAP AI Core orchestration endpoint is reachable.
- Show baseline behavior **without** business context.

Fast engagement tip:
- Run this step and Step 1.6 back-to-back to see the learning "aha" early (ungrounded vs grounded evidence-driven output).

This creates an explicit before/after comparison so participants see the value of Grounding.

In [24]:
baseline_response = orchestration_service.run(
    config=OrchestrationConfig(
        template=Template(messages=[
            SystemMessage("You are a procurement analyst."),
            UserMessage("What are the payment terms in the Techtronix contract?")
        ]),
        llm=LLM(name=MODEL_NAME)
    )
)

print("Baseline response (no grounding):")
print("-" * 70)
print(baseline_response.orchestration_result.choices[0].message.content)
print("-" * 70)


Baseline response (no grounding):
----------------------------------------------------------------------
I don't have access to the Techtronix contract or any specific contract documents in my current context. To help you find the payment terms, I would need:

1. **Access to the contract document** - either uploaded or shared with me
2. **The specific contract identifier** - if there are multiple Techtronix contracts

If you can provide the contract document or relevant excerpts, I can help you identify:
- Payment schedules (e.g., Net 30, Net 60)
- Payment milestones
- Advance payment requirements
- Late payment penalties
- Currency and payment methods
- Invoice submission requirements

Could you share the contract document or let me know where I can access it?
----------------------------------------------------------------------


### Step 1.5 - Grounding Retrieval Helper

This helper calls SAP Document Grounding retrieval directly:
- Endpoint: `/lm/document-grounding/retrieval/search`
- Auth: AI Core OAuth token (client credentials)
- Scope: your configured resource group and repository

Why we use this in the workshop:
- Participants can see retrieval mechanics and provenance.
- It keeps the flow transparent and debuggable for learning.

Return values:
1. Formatted context for prompt augmentation
2. Source list for evidence traceability
3. Chunk count for observability

> **📌 BTP Design Decision:** We call the Document Grounding API directly (not build our own RAG pipeline) because SAP manages the vector index, chunking strategy, and embedding model. This means zero infrastructure for the workshop participant — and a production-ready retrieval layer for customers.

In [25]:
class GroundingClient:
    """
    Client for SAP AI Core Document Grounding API.
    
    Handles OAuth token management and retrieval requests with automatic
    token refresh and caching.
    """
    
    def __init__(self, auth_url: str, client_id: str, client_secret: str,
                 base_url: str, resource_group: str):
        self._auth_url = auth_url
        self._client_id = client_id
        self._client_secret = client_secret
        self._base_url = base_url
        self._resource_group = resource_group
        self._access_token: str | None = None
        self._token_expires_at: float = 0.0
    
    def _get_token(self) -> str:
        """Get a valid access token, refreshing if necessary."""
        now = time.time()
        if self._access_token and now < (self._token_expires_at - TOKEN_REFRESH_BUFFER_SECONDS):
            return self._access_token
        
        response = requests.post(
            f"{self._auth_url}/oauth/token",
            data={"grant_type": "client_credentials"},
            auth=(self._client_id, self._client_secret),
            timeout=TOKEN_TIMEOUT_SECONDS,
        )
        response.raise_for_status()
        payload = response.json()
        
        if "access_token" not in payload:
            raise RuntimeError(f"Token response missing 'access_token'. Keys: {list(payload.keys())}")
        
        self._access_token = payload["access_token"]
        self._token_expires_at = now + int(payload.get("expires_in", 600))
        return self._access_token
    
    def _extract_chunks_from_payload(self, payload: dict) -> List[Tuple[str, str]]:
        """
        Extract (source, content) pairs from grounding API response.
        
        Args:
            payload: Raw JSON response from Document Grounding API
            
        Returns:
            List of (source_id, chunk_content) tuples
        """
        chunks = []
        for filter_result in payload.get("results", []):
            for repo_result in filter_result.get("results", []):
                for doc in repo_result.get("dataRepository", {}).get("documents", []):
                    source = self._get_source_from_metadata(doc.get("metadata", []))
                    for chunk in doc.get("chunks", []):
                        content = (chunk.get("content") or "").strip()
                        if content:
                            chunks.append((source, content))
        return chunks
    
    def _get_source_from_metadata(self, metadata: List[dict]) -> str:
        """Extract source ID from document metadata."""
        for item in metadata:
            if item.get("key") == "id":
                values = item.get("value", [])
                if values:
                    return values[0]
        return "unknown"
    
    def retrieve(self, query: str, data_repository_id: str, max_chunks: int = 6) -> Tuple[str, List[str], int]:
        """
        Retrieve grounded context for a query.
        
        Args:
            query: The search query for document retrieval
            data_repository_id: ID of the SAP Document Grounding repository
            max_chunks: Maximum number of chunks to retrieve (default: 6)
            
        Returns:
            Tuple of (formatted_context, unique_sources, chunk_count)
        """
        token = self._get_token()
        
        response = requests.post(
            f"{self._base_url}/lm/document-grounding/retrieval/search",
            json={
                "query": query,
                "filters": [{
                    "id": "supplier_docs",
                    "dataRepositories": [data_repository_id],
                    "dataRepositoryType": "vector"
                }],
                "maxChunkCount": max_chunks
            },
            headers={
                "Authorization": f"Bearer {token}",
                "AI-Resource-Group": self._resource_group
            },
            timeout=HTTP_TIMEOUT_SECONDS,
        )
        response.raise_for_status()
        
        chunks = self._extract_chunks_from_payload(response.json())
        
        if not chunks:
            return (
                "No relevant grounded context was retrieved. "
                "Try rephrasing your query or check that documents are indexed.",
                [],
                0
            )
        
        # Format chunks with source attribution
        formatted_chunks = [f"[Source: {source}]\n{content}" for source, content in chunks]
        context = "\n\n---\n\n".join(formatted_chunks)
        unique_sources = sorted(set(source for source, _ in chunks))
        
        return context, unique_sources, len(chunks)


if "grounding_client" not in globals():
    grounding_client = None

# If Step 1.3 already loaded env vars, auto-initialize GroundingClient here.
if grounding_client is None and all([
    os.getenv("AICORE_AUTH_URL"),
    os.getenv("AICORE_CLIENT_ID"),
    os.getenv("AICORE_CLIENT_SECRET"),
    os.getenv("AICORE_BASE_URL"),
    os.getenv("AICORE_RESOURCE_GROUP"),
]):
    grounding_client = GroundingClient(
        auth_url=os.getenv("AICORE_AUTH_URL"),
        client_id=os.getenv("AICORE_CLIENT_ID"),
        client_secret=os.getenv("AICORE_CLIENT_SECRET"),
        base_url=os.getenv("AICORE_BASE_URL"),
        resource_group=os.getenv("AICORE_RESOURCE_GROUP"),
    )


def retrieve_grounding_context(query: str, max_chunks: int = 6) -> Tuple[str, List[str], int]:
    """
    Retrieve grounded context from SAP Document Grounding.
    
    This is a convenience wrapper around GroundingClient.retrieve() that uses
    the globally configured client and repository.
    
    Args:
        query: The search query for document retrieval
        max_chunks: Maximum number of chunks to retrieve (default: 6)
        
    Returns:
        Tuple of (formatted_context, unique_sources, chunk_count)
        
    Raises:
        RuntimeError: If grounding client is not initialized
    """
    if grounding_client is None:
        raise RuntimeError(
            "Grounding client not initialized. Run Step 1.3 (load .env), then re-run this cell."
        )
    
    return grounding_client.retrieve(query, DATA_REPOSITORY_ID, max_chunks)


print("GroundingClient class and retrieve_grounding_context() ready.")

GroundingClient class and retrieve_grounding_context() ready.


### Step 1.6 - Inspect Retrieved Grounding Chunks

Run this to inspect the raw grounding evidence for a procurement query.

Learning objective:
- Confirm retrieval quality before generation.

Expected output signature:
- Chunk count > 0
- Source list with document IDs
- Supplier-specific terms in context preview

Checkpoint (after running):
- You can explain how retrieval quality impacts answer quality.

In [26]:
sample_query = "Techtronix delivery performance and contract penalty terms"
context, source_list, chunk_count = retrieve_grounding_context(sample_query, max_chunks=5)

print(f"Query: {sample_query}")
print(f"Retrieved chunks: {chunk_count}")
print(f"Sources: {source_list}")
print("Preview:")
print("-" * 70)
print(context[:1800])
print("-" * 70)


Query: Techtronix delivery performance and contract penalty terms
Retrieved chunks: 10
Sources: ['on_time_delivery_report.txt', 'po_with_contacts.txt', 'pulsewave_materials_inc._contract.pdf', 'quality_incidents_with_contacts.txt', 'sigma_electronics_co._contract.pdf', 'supplier_audits.pdf', 'techtronix_components_ltd._contract.pdf']
Preview:
----------------------------------------------------------------------
[Source: techtronix_components_ltd._contract.pdf]
Supplier Contract
This contract is made between BestRun Technologies and Techtronix Components Ltd. on
2025-01-01.
Scope: Supply of mechanical components including smart sensor housings and motion detector
assemblies.
Delivery Terms: Techtronix Components Ltd. shall deliver goods within 30 days of purchase order
issuance.
Quality Requirements: Products must meet ISO 9001 standards applicable to IoT device
components.
Late Delivery Penalty: For deliveries more than 3 days late, a penalty of 5% of the order value will
be applied f

---
### 🏗️ Build: Deterministic Grounded Q&A

### Step 1.7 - Deterministic Grounded Query Function

This function is your standard **non-agent** enterprise pattern:
1. retrieve grounded context from SAP Grounding
2. augment the prompt
3. generate answer through SAP OrchestrationService

Use this when workflow is fixed and deterministic.

In [27]:
def grounded_query(
    question: str,
    system_role: str = "You are a procurement analyst.",
    max_chunks: int = 6
) -> Dict[str, Any]:
    """
    Execute a grounded Q&A query using SAP Document Grounding and OrchestrationService.
    
    This is the standard deterministic pattern for enterprise Q&A:
    1. Retrieve relevant chunks from indexed documents
    2. Augment the prompt with grounded context
    3. Generate answer via LLM
    
    Args:
        question: The business question to answer
        system_role: System prompt defining the AI's persona (default: procurement analyst)
        max_chunks: Maximum chunks to retrieve for context (default: 6)
        
    Returns:
        Dict containing:
            - question: The original question
            - answer: The generated response
            - sources: List of source document IDs used
            - chunk_count: Number of chunks retrieved
            - grounded: Boolean indicating if context was found
    """
    context, sources, chunk_count = retrieve_grounding_context(question, max_chunks=max_chunks)
    
    # Check if we got meaningful context
    is_grounded = chunk_count > 0
    
    if not is_grounded:
        # Provide clear feedback when no context is found
        user_prompt = (
            "No relevant documents were found for this query. "
            "Please indicate that you cannot answer based on the available grounded context, "
            "and suggest the user try rephrasing their question.\n\n"
            f"Question: {question}"
        )
    else:
        user_prompt = (
            "Use only the grounded context below. If evidence is missing, say so.\n\n"
            f"Grounded context:\n{context}\n\n"
            f"Question: {question}"
        )
    
    response = orchestration_service.run(
        config=OrchestrationConfig(
            template=Template(messages=[
                SystemMessage(system_role),
                UserMessage(user_prompt),
            ]),
            llm=LLM(name=MODEL_NAME),
        )
    )

    answer = response.orchestration_result.choices[0].message.content
    
    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "chunk_count": chunk_count,
        "grounded": is_grounded,
    }


print("grounded_query() ready.")

grounded_query() ready.


### Step 1.8 - Run Deterministic Grounded Examples

> **🎯 What We're Really Asking the AI:**
> These aren't generic questions — they're the exact questions a procurement analyst would ask during a supplier review. Each one maps to a real business decision point:
> - *"What are the payment and penalty terms?"* → Can we negotiate better terms at renewal?
> - *"Summarize delivery performance"* → Is the supplier reliable enough to keep?

These examples are intentionally single-path questions where agent orchestration is unnecessary.

This reinforces the architecture principle: **if one retrieval + one generation solves it, do not add agent complexity.**

In [28]:
essential_questions = [
    "What are the payment and late delivery penalty terms in the Techtronix contract?",
    "Summarize Techtronix delivery performance based on available reports.",
]

for q in essential_questions:
    result = grounded_query(q)
    print("=" * 80)
    print(f"Question: {result['question']}")
    print(f"Chunk count: {result['chunk_count']}")
    print(f"Sources: {result['sources']}")
    print("Answer:")
    print(result["answer"])


Question: What are the payment and late delivery penalty terms in the Techtronix contract?
Chunk count: 10
Sources: ['on_time_delivery_report.txt', 'po_with_contacts.txt', 'pulsewave_materials_inc._contract.pdf', 'quality_incidents_with_contacts.txt', 'sigma_electronics_co._contract.pdf', 'supplier_audits.pdf', 'techtronix_components_ltd._contract.pdf']
Answer:
Based on the grounded context from the Techtronix Components Ltd. contract:

## Late Delivery Penalty Terms:
For deliveries more than 3 days late, a penalty of **5% of the order value will be applied for each week of delay**.

## Payment Terms:
**No payment terms are specified** in the provided contract document. The contract includes delivery terms, quality requirements, late delivery penalties, termination clauses, and governing law, but does not contain information about payment terms, payment schedules, or payment methods.
Question: Summarize Techtronix delivery performance based on available reports.
Chunk count: 10
Sources

### Step 1.9 - Exercise (Essential)

Try these deterministic questions with `grounded_query()`:
- What is the Techtronix contract termination notice period?
- What quality issues are recorded for Techtronix?
- Compare late delivery penalty clauses across Techtronix, Sigma, and PulseWave.

Expected participant outcome:
- grounded answers
- visible source references
- clear understanding of deterministic grounding pattern


In [29]:
your_question = "What is the Techtronix contract termination notice period?"
exercise_result = grounded_query(your_question)

print(f"Question: {your_question}")
print(f"Sources used: {exercise_result['sources']}")
print(exercise_result["answer"])


Question: What is the Techtronix contract termination notice period?
Sources used: ['on_time_delivery_report.txt', 'po_with_contacts.txt', 'pulsewave_materials_inc._contract.pdf', 'quality_incidents_with_contacts.txt', 'sigma_electronics_co._contract.pdf', 'supplier_audits.pdf', 'techtronix_components_ltd._contract.pdf']
Based on the grounded context provided:

According to the Techtronix Components Ltd. contract (dated 2025-01-01), the termination notice period is **30 days**.

Specifically, the contract states: "Either party may terminate this contract with 30 days' notice for cause."


---
### Decide: When to Use an Agent

### Step 1.10 - Decision Framework: When to Use Agent vs Not

| Dimension | Grounded Q&A (`grounded_query`) | Agent (ReAct) |
| --- | --- | --- |
| **When to use** | Single question, fixed process | Multi-criteria, adaptive reasoning |
| **BTP services** | Grounding + Orchestration | Same + tool layer + trace |
| **Complexity** | Low | Medium-High |
| **Latency** | 1 retrieval + 1 LLM call | N tool calls + N LLM calls |
| **Governance** | Answer + sources | Trace + approval gate |
| **Example** | "What is the penalty clause?" | "Should we renew this supplier?" |

Use `grounded_query()` when:
- single question
- fixed process
- no branching decisions

Use an agent when:
- findings change next actions
- multiple specialized tools are needed
- recommendation quality depends on iterative synthesis

Checkpoint (before Part 2):
- You can justify, in one sentence, why renewal is agentic and not deterministic.

In [30]:
def decide_pattern(task_description: str) -> str:
    text = task_description.lower()
    agent_signals = [
        "recommend",
        "should we",
        "compare",
        "trade-off",
        "multiple",
        "if",
        "decision",
        "renew",
    ]
    matched = [s for s in agent_signals if s in text]
    if len(matched) >= 2:
        return f"Agent recommended. Signals: {matched}"
    return "Deterministic grounded_query recommended."

examples = [
    "What is the late delivery penalty for Techtronix?",
    "Should we renew Techtronix based on delivery, quality, and contract risk?",
]
for e in examples:
    print(f"Task: {e}")
    print(f"Decision: {decide_pattern(e)}")
    print("-" * 70)


Task: What is the late delivery penalty for Techtronix?
Decision: Deterministic grounded_query recommended.
----------------------------------------------------------------------
Task: Should we renew Techtronix based on delivery, quality, and contract risk?
Decision: Agent recommended. Signals: ['should we', 'renew']
----------------------------------------------------------------------


---

> ✅ **Checkpoint — Part 1 Complete**
>
> You now have a working grounded Q&A pipeline. You can answer single business questions using SAP Document Grounding + OrchestrationService — and you understand *when* this deterministic pattern is sufficient.
>
> **In Part 2** (optional advanced), we move to multi-step, non-deterministic decisions where an agent adds real value: the supplier renewal recommendation.

## Part 2 (Optional Advanced): ReAct Agent on SAP AI Core

This section is optional advanced content.

Goal: demonstrate a pro-code ReAct pattern on SAP services for non-deterministic business decisions.

### Top 5 Agent Architecture Best Practices in This Section

| # | Best Practice | How It's Applied Here |
| --- | --- | --- |
| 1 | **Decision-first architecture** | Use agent only for non-deterministic tasks |
| 2 | **Strict tool contracts** | Typed inputs and controlled outputs |
| 3 | **Grounded evidence and provenance** | Every tool result includes sources |
| 4 | **Guardrails and structured outputs** | JSON schema instead of fragile free text parsing |
| 5 | **Observability and human oversight** | Step traces plus approval checkpoint |

### Step 2.1a - Define Business Agent Tools with Evidence Outputs

Each tool wraps a focused business capability and uses SAP services under the hood:
- SAP Document Grounding retrieval for evidence acquisition
- SAP OrchestrationService for role-specific synthesis

Why this is important for BTP adoption:
- modular design with clear responsibilities
- evidence-backed outputs suitable for enterprise review
- SAP-native runtime path from prototype to production

Expected output signature (after running Step 2.1a code):
- Tool registry contains 3 business tools for delivery, quality, and contract analysis.

BTP design decision: each business tool wraps SAP services (not generic OpenAI calls) so execution stays within SAP AI Core boundaries (enterprise auth, resource-group isolation, audit trail).

In [31]:
# --- Tool Configuration ---
TOOL_CONFIGS = {
    TOOL_DELIVERY_PERFORMANCE: {
        "system_role": "You are a supply chain analyst. Return 3-5 bullet points.",
        "query_template": "{supplier_name} delivery report on-time late penalty",
        "question_template": "Summarize delivery performance for {supplier_name}.",
    },
    TOOL_QUALITY_INCIDENTS: {
        "system_role": "You are a quality analyst. Return 3-5 bullet points.",
        "query_template": "{supplier_name} quality incidents audit score defects",
        "question_template": "Summarize quality incidents and quality score for {supplier_name}.",
    },
    TOOL_CONTRACT_TERMS: {
        "system_role": "You are a contract analyst. Cite exact clauses when present.",
        "query_template": "{supplier_name} contract {topic} clauses",
        "question_template": "Extract contract terms for {supplier_name} about: {topic}.",
    },
}


def _validate_supplier_name(supplier_name: str) -> None:
    """
    Validate supplier name input.
    
    Raises:
        ValueError: If supplier_name is empty or invalid
    """
    if not supplier_name or not isinstance(supplier_name, str):
        raise ValueError("supplier_name must be a non-empty string")
    if len(supplier_name.strip()) < 2:
        raise ValueError(f"supplier_name too short: '{supplier_name}'")


def _summarize_with_role(system_role: str, question: str, context: str) -> str:
    """
    Generate a summary using OrchestrationService with a specific role.
    
    Args:
        system_role: The persona/role for the system prompt
        question: The question to answer
        context: The grounded context to use
        
    Returns:
        The generated summary text
    """
    response = orchestration_service.run(
        config=OrchestrationConfig(
            template=Template(messages=[
                SystemMessage(system_role),
                UserMessage(
                    "Use only the grounded context below and be concise.\n\n"
                    f"Grounded context:\n{context}\n\nQuestion: {question}"
                ),
            ]),
            llm=LLM(name=MODEL_NAME),
        )
    )
    return response.orchestration_result.choices[0].message.content


def _create_tool_result(
    tool_name: str,
    supplier_name: str,
    summary: str,
    sources: List[str],
    chunk_count: int,
    **extra_fields
) -> Dict[str, Any]:
    """Create a standardized tool result dictionary."""
    result = {
        "tool": tool_name,
        "supplier_name": supplier_name,
        "summary": summary,
        "sources": sources,
        "chunk_count": chunk_count,
    }
    result.update(extra_fields)
    return result


def query_delivery_performance(supplier_name: str) -> Dict[str, Any]:
    """
    Query delivery performance data for a supplier.
    
    Args:
        supplier_name: The supplier name to query (e.g., "Techtronix Components Ltd.")
        
    Returns:
        Dict with tool name, supplier, summary, sources, and chunk count
        
    Raises:
        ValueError: If supplier_name is invalid
    """
    _validate_supplier_name(supplier_name)
    
    config = TOOL_CONFIGS[TOOL_DELIVERY_PERFORMANCE]
    query = config["query_template"].format(supplier_name=supplier_name)
    question = config["question_template"].format(supplier_name=supplier_name)
    
    context, sources, chunk_count = retrieve_grounding_context(query, max_chunks=5)
    summary = _summarize_with_role(config["system_role"], question, context)
    
    return _create_tool_result(
        TOOL_DELIVERY_PERFORMANCE, supplier_name, summary, sources, chunk_count
    )


def query_quality_incidents(supplier_name: str) -> Dict[str, Any]:
    """
    Query quality incidents and audit scores for a supplier.
    
    Args:
        supplier_name: The supplier name to query (e.g., "Techtronix Components Ltd.")
        
    Returns:
        Dict with tool name, supplier, summary, sources, and chunk count
        
    Raises:
        ValueError: If supplier_name is invalid
    """
    _validate_supplier_name(supplier_name)
    
    config = TOOL_CONFIGS[TOOL_QUALITY_INCIDENTS]
    query = config["query_template"].format(supplier_name=supplier_name)
    question = config["question_template"].format(supplier_name=supplier_name)
    
    context, sources, chunk_count = retrieve_grounding_context(query, max_chunks=5)
    summary = _summarize_with_role(config["system_role"], question, context)
    
    return _create_tool_result(
        TOOL_QUALITY_INCIDENTS, supplier_name, summary, sources, chunk_count
    )


def query_contract_terms(supplier_name: str, topic: str = "general") -> Dict[str, Any]:
    """
    Query contract terms for a supplier on a specific topic.
    
    Args:
        supplier_name: The supplier name to query (e.g., "Techtronix Components Ltd.")
        topic: The contract topic to focus on (e.g., "penalties", "termination")
        
    Returns:
        Dict with tool name, supplier, topic, summary, sources, and chunk count
        
    Raises:
        ValueError: If supplier_name is invalid
    """
    _validate_supplier_name(supplier_name)
    
    if not topic or not isinstance(topic, str):
        topic = "general"
    
    config = TOOL_CONFIGS[TOOL_CONTRACT_TERMS]
    query = config["query_template"].format(supplier_name=supplier_name, topic=topic)
    question = config["question_template"].format(supplier_name=supplier_name, topic=topic)
    
    context, sources, chunk_count = retrieve_grounding_context(query, max_chunks=5)
    summary = _summarize_with_role(config["system_role"], question, context)
    
    return _create_tool_result(
        TOOL_CONTRACT_TERMS, supplier_name, summary, sources, chunk_count, topic=topic
    )


# Tool registry using constants
TOOLS = {
    TOOL_DELIVERY_PERFORMANCE: query_delivery_performance,
    TOOL_QUALITY_INCIDENTS: query_quality_incidents,
    TOOL_CONTRACT_TERMS: query_contract_terms,
}

print(f"Agent tools ready: {list(TOOLS.keys())}")

Agent tools ready: ['query_delivery_performance', 'query_quality_incidents', 'query_contract_terms']


### Step 2.1b - Register Presentation Tool

This step registers `render_result_dashboard` into the same tool registry used by the agent.

Why this matters:
- one explicit tool registry for all callable tools
- clear separation between reasoning tools and presentation tool
- easy migration to a trusted visualization/MCP backend later

Expected output signature (after running Step 2.1b code):
- Tool registry includes `render_result_dashboard` with business tools.

In [32]:
import html
from IPython.display import display, HTML

# Presentation tool (kept with tool definitions for cleaner workshop architecture)
TOOL_RENDER_DASHBOARD = "render_result_dashboard"


def _strip_surrogates(text: str) -> str:
    """Remove invalid UTF-16 surrogate code points that break Jupyter serialization."""
    return "".join(ch for ch in text if not (0xD800 <= ord(ch) <= 0xDFFF))


def _safe_str(value: Any) -> str:
    """Convert any value to a display-safe string."""
    return _strip_surrogates(str(value))


def _safe_html(value: Any) -> str:
    """Convert any value to escaped, surrogate-safe HTML text."""
    return html.escape(_safe_str(value))


def display_agent_result(result: Dict[str, Any]) -> None:
    """Render the agent result as a styled HTML dashboard with UTF-safe sanitization."""
    confidence = _safe_str(result.get("confidence", "unknown"))
    conf_colors = {"high": "#2e7d32", "medium": "#f57f17", "low": "#c62828", "unknown": "#757575"}
    conf_color = conf_colors.get(confidence, conf_colors["unknown"])

    approval = _safe_str(result.get("approval_state", "unknown"))
    approval_labels = {
        "pending_human_approval": ("Pending Human Approval", "#e65100"),
        "approved": ("Approved", "#2e7d32"),
        "incomplete": ("Incomplete", "#c62828"),
    }
    approval_text, approval_color = approval_labels.get(approval, (approval, "#757575"))

    tool_history = result.get("tool_history", [])
    evidence_rows = ""
    for i, th in enumerate(tool_history, 1):
        tool_name = _safe_str(th.get("tool", ""))
        args_items = th.get("args", {})
        args_str = ", ".join(f"{_safe_str(k)}={_safe_str(v)}" for k, v in args_items.items())

        sources = []
        for t in result.get("trace", []):
            if _safe_str(t.get("tool", "")) == tool_name and _safe_str(t.get("type", "")) == "action":
                sources = t.get("result_sources", [])
                break

        sources_str = ", ".join(_safe_str(s) for s in sources) if sources else "-"
        evidence_rows += (
            f'<tr><td style="padding:6px 12px;border-bottom:1px solid #e0e0e0;">{i}</td>'
            f'<td style="padding:6px 12px;border-bottom:1px solid #e0e0e0;"><code>{_safe_html(tool_name)}</code></td>'
            f'<td style="padding:6px 12px;border-bottom:1px solid #e0e0e0;">{_safe_html(args_str)}</td>'
            f'<td style="padding:6px 12px;border-bottom:1px solid #e0e0e0;font-size:0.85em;">{_safe_html(sources_str)}</td></tr>'
        )

    type_colors = {"action": "#1565c0", "final": "#2e7d32", "guardrail": "#e65100", "retry": "#c62828"}
    trace_rows = ""
    for t in result.get("trace", []):
        step_type = _safe_str(t.get("type", ""))
        color = type_colors.get(step_type, "#757575")
        badge = f'<span style="background:{color};color:#fff;padding:2px 8px;border-radius:4px;font-size:0.8em;">{_safe_html(step_type)}</span>'
        tool_value = _safe_str(t.get("tool", ""))
        tool_cell = f'<code>{_safe_html(tool_value)}</code>' if tool_value else "-"
        thought = _safe_str(t.get("thought", ""))[:120]
        latency = int(t.get("latency_ms", 0) or 0)
        step_num = _safe_html(t.get("step", ""))

        trace_rows += (
            f'<tr>'
            f'<td style="padding:6px 10px;border-bottom:1px solid #e0e0e0;text-align:center;">{step_num}</td>'
            f'<td style="padding:6px 10px;border-bottom:1px solid #e0e0e0;">{badge}</td>'
            f'<td style="padding:6px 10px;border-bottom:1px solid #e0e0e0;">{tool_cell}</td>'
            f'<td style="padding:6px 10px;border-bottom:1px solid #e0e0e0;font-size:0.85em;">{_safe_html(thought)}</td>'
            f'<td style="padding:6px 10px;border-bottom:1px solid #e0e0e0;text-align:right;">{latency:,} ms</td>'
            f'</tr>'
        )

    answer_text = _safe_html(result.get("answer", "No answer"))
    html_content = f"""
    <div style="font-family:system-ui,-apple-system,sans-serif;max-width:900px;">
      <div style="border:1px solid #bdbdbd;border-radius:8px;padding:20px 24px;margin-bottom:20px;
                  background:linear-gradient(135deg,#fafafa 0%,#f5f5f5 100%);">
        <div style="display:flex;align-items:center;gap:12px;margin-bottom:12px;">
          <span style="font-size:1.4em;font-weight:700;">Agent Recommendation</span>
          <span style="background:{conf_color};color:#fff;padding:3px 10px;border-radius:4px;
                       font-size:0.85em;font-weight:600;">Confidence: {_safe_html(confidence)}</span>
          <span style="background:{approval_color};color:#fff;padding:3px 10px;border-radius:4px;
                       font-size:0.85em;font-weight:600;">{_safe_html(approval_text)}</span>
        </div>
        <div style="font-size:0.95em;line-height:1.6;white-space:pre-wrap;padding:12px 16px;
                    background:#fff;border-radius:6px;border:1px solid #e0e0e0;">{answer_text}</div>
      </div>
      <div style="margin-bottom:20px;">
        <div style="font-size:1.1em;font-weight:600;margin-bottom:8px;">Evidence Gathered ({len(tool_history)} tool calls)</div>
        <table style="border-collapse:collapse;width:100%;background:#fff;border:1px solid #e0e0e0;border-radius:6px;">
          <thead><tr style="background:#eeeeee;">
            <th style="padding:8px 12px;text-align:left;border-bottom:2px solid #bdbdbd;">#</th>
            <th style="padding:8px 12px;text-align:left;border-bottom:2px solid #bdbdbd;">Tool</th>
            <th style="padding:8px 12px;text-align:left;border-bottom:2px solid #bdbdbd;">Arguments</th>
            <th style="padding:8px 12px;text-align:left;border-bottom:2px solid #bdbdbd;">Sources</th>
          </tr></thead>
          <tbody>{evidence_rows}</tbody>
        </table>
      </div>
      <div style="margin-bottom:20px;">
        <div style="font-size:1.1em;font-weight:600;margin-bottom:8px;">Reasoning Trace</div>
        <table style="border-collapse:collapse;width:100%;background:#fff;border:1px solid #e0e0e0;border-radius:6px;">
          <thead><tr style="background:#eeeeee;">
            <th style="padding:8px 10px;text-align:center;border-bottom:2px solid #bdbdbd;">Step</th>
            <th style="padding:8px 10px;text-align:left;border-bottom:2px solid #bdbdbd;">Type</th>
            <th style="padding:8px 10px;text-align:left;border-bottom:2px solid #bdbdbd;">Tool</th>
            <th style="padding:8px 10px;text-align:left;border-bottom:2px solid #bdbdbd;">Thought</th>
            <th style="padding:8px 10px;text-align:right;border-bottom:2px solid #bdbdbd;">Latency</th>
          </tr></thead>
          <tbody>{trace_rows}</tbody>
        </table>
      </div>
    </div>
    """
    display(HTML(_strip_surrogates(html_content)))


def render_result_dashboard(
    answer: str,
    confidence: str = "medium",
    approval_state: str = "pending_human_approval",
    tool_history: List[Dict[str, Any]] | None = None,
    trace: List[Dict[str, Any]] | None = None,
) -> Dict[str, Any]:
    """Tool: render recommendation dashboard from agent-provided args and runtime context."""
    payload = {
        "answer": answer,
        "confidence": confidence,
        "approval_state": approval_state,
        "tool_history": tool_history or [],
        "trace": trace or [],
    }
    display_agent_result(payload)
    return {
        "tool": TOOL_RENDER_DASHBOARD,
        "status": "rendered",
        "confidence": confidence,
        "approval_state": approval_state,
        "rendered_with_trace": bool(trace),
    }


TOOLS[TOOL_RENDER_DASHBOARD] = render_result_dashboard

print(f"Presentation tool registered with tool set: {TOOL_RENDER_DASHBOARD}")

Presentation tool registered with tool set: render_result_dashboard


### Step 2.2 - ReAct Prompt and Structured Action Parser

This section enforces strict JSON ReAct outputs.

Why JSON format:
- reliable parsing
- simple validation/retry handling
- production-aligned orchestration behavior

### ReAct Flow Diagram

```text
User decision question
        |
        v
+-------------------------------+
| ReAct controller (LLM)        |
| chooses next action           |
+-------------------------------+
        |
        | ACTION + ARGS (JSON)
        v
+-------------------------------+
| Tool layer                    |
| - delivery performance tool   |
| - quality incidents tool      |
| - contract terms tool         |
| - result dashboard tool       |
+-------------------------------+
        |
        | tool result (JSON)
        v
controller decides next step
        |
        +--> loop until FINAL
```

In [33]:
AGENT_SYSTEM_PROMPT = """
You are a procurement decision agent for BestRun Technologies.

Rules:
- You must use tools. Do not answer from prior knowledge.
- Think step by step and choose tools based on evidence gaps.
- Return ONLY one valid JSON object. No prose, no markdown, no code fences, no extra text.

For tool calls:
{
  "type": "action",
  "thought": "short reason",
  "tool": "query_delivery_performance | query_quality_incidents | query_contract_terms | render_result_dashboard",
  "args": {"...": "tool-specific arguments"}
}

For final answer:
{
  "type": "final",
  "thought": "short synthesis",
  "answer": "final recommendation",
  "confidence": "low|medium|high"
}

When to use `render_result_dashboard`:
- Use it once after you have enough evidence and a draft recommendation,
  if the user asks to present/visualize/show the recommendation.
- Pass concise args: `answer`, `confidence`, `approval_state`.
- After calling it, still return a normal `final` response.
""".strip()


# Enable verbose parsing logs for debugging (set to True in workshop if issues arise)
PARSE_DEBUG = False


def _extract_first_json_object(text: str) -> str | None:
    """Extract the first balanced JSON object from arbitrary text."""
    start = text.find("{")
    if start == -1:
        return None

    in_string = False
    escape = False
    depth = 0

    for i in range(start, len(text)):
        ch = text[i]

        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]

    return None


def parse_agent_json(text: str) -> Dict[str, Any]:
    """
    Parse agent response text to extract JSON action or final answer.

    Robust against extra content by extracting the first balanced JSON object.
    """
    if PARSE_DEBUG:
        print(f"[PARSE_DEBUG] Input length: {len(text)} chars")
        print(f"[PARSE_DEBUG] Input preview: {text[:200]}...")

    snippet = _extract_first_json_object(text)
    if not snippet:
        error_msg = "No JSON object found in response."
        if PARSE_DEBUG:
            print(f"[PARSE_DEBUG] {error_msg}")
        return {
            "type": RETRY_TYPE,
            "error": error_msg,
            "raw_preview": text[:120] if text else "(empty)",
        }

    if PARSE_DEBUG:
        print(f"[PARSE_DEBUG] Extracted JSON: {snippet[:200]}...")

    try:
        payload = json.loads(snippet)
    except json.JSONDecodeError as exc:
        error_msg = f"Invalid JSON at position {exc.pos}: {exc.msg}"
        if PARSE_DEBUG:
            print(f"[PARSE_DEBUG] {error_msg}")
        return {
            "type": RETRY_TYPE,
            "error": error_msg,
            "raw_preview": snippet[:120],
        }

    response_type = payload.get("type")
    if response_type not in {ACTION_TYPE, FINAL_TYPE}:
        error_msg = f"Invalid 'type' value: '{response_type}'. Expected '{ACTION_TYPE}' or '{FINAL_TYPE}'."
        if PARSE_DEBUG:
            print(f"[PARSE_DEBUG] {error_msg}")
        return {
            "type": RETRY_TYPE,
            "error": error_msg,
        }

    if response_type == ACTION_TYPE and "tool" not in payload:
        return {
            "type": RETRY_TYPE,
            "error": "Action missing required 'tool' field.",
        }

    if response_type == FINAL_TYPE and "answer" not in payload:
        return {
            "type": RETRY_TYPE,
            "error": "Final answer missing required 'answer' field.",
        }

    if PARSE_DEBUG:
        print(f"[PARSE_DEBUG] Successfully parsed type='{response_type}'")

    return payload


print("Agent parser ready (set PARSE_DEBUG=True for verbose logging).")

Agent parser ready (set PARSE_DEBUG=True for verbose logging).


### Step 2.3 - Run ReAct Agent (with Trace + Approval Gate)

This runtime loop demonstrates enterprise controls:
- minimum tool-usage guardrail
- schema validation and retry behavior
- step-level observability trace
- explicit human approval state before operational action


In [34]:
def _execute_tool(tool_name: str, args: Dict[str, Any]) -> Dict[str, Any]:
    """Execute an agent tool by name with given arguments."""
    if tool_name not in TOOLS:
        return {"error": f"Unknown tool: '{tool_name}'. Available: {list(TOOLS.keys())}"}
    
    try:
        return TOOLS[tool_name](**args)
    except ValueError as e:
        return {"error": f"Validation error in {tool_name}: {e}"}
    except Exception as e:
        return {"error": f"Tool '{tool_name}' failed: {type(e).__name__}: {e}"}


def _create_trace_entry(
    step: int,
    entry_type: str,
    thought: str,
    latency_ms: int,
    tool: str | None = None,
    args: Dict[str, Any] | None = None,
    result_sources: List[str] | None = None,
) -> Dict[str, Any]:
    """Create a standardized trace entry for observability."""
    return {
        "step": step,
        "type": entry_type,
        "thought": thought,
        "tool": tool,
        "args": args,
        "latency_ms": latency_ms,
        "result_sources": result_sources or [],
    }


def _handle_action(
    parsed: Dict[str, Any],
    step: int,
    elapsed_ms: int,
    messages: List,
    tool_history: List[Dict[str, Any]],
    trace: List[Dict[str, Any]],
) -> None:
    """Handle an action response from the agent."""
    tool_name = parsed.get("tool")
    args = parsed.get("args", {})
    thought = parsed.get("thought", "")

    # Inject runtime context for presentation tool so it can render evidence + trace.
    if tool_name == "render_result_dashboard":
        args = dict(args)
        args.setdefault("tool_history", list(tool_history))
        args.setdefault("trace", list(trace))
    
    tool_result = _execute_tool(tool_name, args)
    
    if "error" not in tool_result:
        tool_history.append({"tool": tool_name, "args": args})
    
    trace.append(_create_trace_entry(
        step=step,
        entry_type=ACTION_TYPE,
        thought=thought,
        latency_ms=elapsed_ms,
        tool=tool_name,
        args=args,
        result_sources=tool_result.get("sources", []),
    ))
    
    print(f"Step {step}: ACTION -> {tool_name}({args})")
    
    messages.append(UserMessage(
        f"Tool result (JSON):\n{json.dumps(tool_result, ensure_ascii=True)}\n\n"
        f"Tools used so far: {len(tool_history)}."
    ))


def _handle_final(
    parsed: Dict[str, Any],
    step: int,
    elapsed_ms: int,
    trace: List[Dict[str, Any]],
    require_human_approval: bool,
) -> Dict[str, Any]:
    """Handle a final answer response from the agent."""
    final_answer = parsed.get("answer", "")
    confidence = parsed.get("confidence", "unknown")
    thought = parsed.get("thought", "")
    
    trace.append(_create_trace_entry(
        step=step,
        entry_type=FINAL_TYPE,
        thought=thought,
        latency_ms=elapsed_ms,
    ))
    
    print(f"Step {step}: FINAL reached (confidence={confidence})")
    
    return {
        "answer": final_answer,
        "confidence": confidence,
        "approval_state": "pending_human_approval" if require_human_approval else "approved",
    }


def _handle_guardrail(
    step: int,
    elapsed_ms: int,
    tool_count: int,
    min_tools: int,
    messages: List,
    trace: List[Dict[str, Any]],
) -> None:
    """Handle guardrail violation (not enough tools used)."""
    trace.append(_create_trace_entry(
        step=step,
        entry_type="guardrail",
        thought=f"Minimum tool count not met ({tool_count}/{min_tools}).",
        latency_ms=elapsed_ms,
    ))
    
    print(f"Step {step}: GUARDRAIL -> requires at least {min_tools} tools")
    messages.append(UserMessage(
        f"You used {tool_count} tool(s). Use at least {min_tools} tools before final answer."
    ))


def _handle_retry(
    parsed: Dict[str, Any],
    step: int,
    elapsed_ms: int,
    messages: List,
    trace: List[Dict[str, Any]],
) -> None:
    """Handle a parse failure requiring retry."""
    error_msg = parsed.get("error", "format error")
    
    trace.append(_create_trace_entry(
        step=step,
        entry_type=RETRY_TYPE,
        thought=error_msg,
        latency_ms=elapsed_ms,
    ))
    
    print(f"Step {step}: RETRY -> {error_msg}")
    messages.append(UserMessage(
        "Return only one valid JSON object in the required schema."
    ))


def run_react_agent(
    question: str,
    min_tools: int = 2,
    max_steps: int = 6,
    require_human_approval: bool = True,
) -> Dict[str, Any]:
    """Run a ReAct agent loop to answer a complex business question."""
    messages = [SystemMessage(AGENT_SYSTEM_PROMPT), UserMessage(question)]
    tool_history: List[Dict[str, Any]] = []
    trace: List[Dict[str, Any]] = []

    print("=" * 80)
    print("REACT AGENT START")
    print(f"Question: {question}")
    print(f"Config: min_tools={min_tools}, max_steps={max_steps}")
    print("=" * 80)

    for step in range(1, max_steps + 1):
        step_start = time.time()
        
        response = orchestration_service.run(
            config=OrchestrationConfig(
                template=Template(messages=messages),
                llm=LLM(name=MODEL_NAME),
            )
        )
        raw = response.orchestration_result.choices[0].message.content
        parsed = parse_agent_json(raw)
        elapsed_ms = int((time.time() - step_start) * 1000)

        if parsed.get("type") == ACTION_TYPE:
            _handle_action(parsed, step, elapsed_ms, messages, tool_history, trace)
            continue

        if parsed.get("type") == FINAL_TYPE:
            if len(tool_history) < min_tools:
                _handle_guardrail(step, elapsed_ms, len(tool_history), min_tools, messages, trace)
                continue
            
            final_result = _handle_final(parsed, step, elapsed_ms, trace, require_human_approval)
            return {
                "question": question,
                "answer": final_result["answer"],
                "confidence": final_result["confidence"],
                "tool_history": tool_history,
                "trace": trace,
                "approval_state": final_result["approval_state"],
            }

        _handle_retry(parsed, step, elapsed_ms, messages, trace)

    print(f"Agent reached max steps ({max_steps}) without final answer.")
    return {
        "question": question,
        "answer": "Agent reached max steps without final answer.",
        "confidence": "low",
        "tool_history": tool_history,
        "trace": trace,
        "approval_state": "incomplete",
    }


print("run_react_agent() and helper functions ready.")

run_react_agent() and helper functions ready.


### Step 2.4 - Run One Advanced Agent Scenario

Advanced scenario:
- Should BestRun renew Techtronix based on delivery, quality, and contract evidence?

This is non-deterministic and requires selective multi-tool reasoning.

Expected output signature:
- 2-3 business tool calls before final answer
- One `render_result_dashboard` call when visualization is requested
- Final answer with confidence and `pending_human_approval`

Checkpoint (after running):
- You can explain how intermediate evidence changed tool choice or confidence.

In [35]:
agent_result = run_react_agent(
    question=(
        "Should BestRun Technologies renew the Techtronix Components Ltd. contract? "
        "Use delivery performance, quality incidents, and contract terms. "
        "Present the recommendation in a dashboard for executive review, then provide final rationale."
    ),
    min_tools=2,
    max_steps=7,
    require_human_approval=True,
)

print("\nFinal recommendation (text record):")
print("-" * 80)
print(agent_result.get("answer", "No answer"))
print("-" * 80)
print(f"Confidence: {agent_result.get('confidence', 'unknown')}")
print(f"Approval state: {agent_result.get('approval_state', 'unknown')}")
print(f"Tools used: {agent_result.get('tool_history', [])}")

REACT AGENT START
Question: Should BestRun Technologies renew the Techtronix Components Ltd. contract? Use delivery performance, quality incidents, and contract terms. Present the recommendation in a dashboard for executive review, then provide final rationale.
Config: min_tools=2, max_steps=7
Step 1: ACTION -> query_delivery_performance({'supplier_name': 'Techtronix Components Ltd.'})
Step 2: ACTION -> query_quality_incidents({'supplier_name': 'Techtronix Components Ltd.'})
Step 3: ACTION -> query_contract_terms({'supplier_name': 'Techtronix Components Ltd.'})


Step 4: ACTION -> render_result_dashboard({'answer': 'Conditional Renewal Recommended: Techtronix Components Ltd. demonstrates strong delivery reliability (90/100) but has recurring quality issues (85/100) with mechanical precision. Recommend renewal with enhanced quality controls and quarterly audits for 6-month probationary period.', 'confidence': 'medium', 'approval_state': 'conditional', 'tool_history': [{'tool': 'query_delivery_performance', 'args': {'supplier_name': 'Techtronix Components Ltd.'}}, {'tool': 'query_quality_incidents', 'args': {'supplier_name': 'Techtronix Components Ltd.'}}, {'tool': 'query_contract_terms', 'args': {'supplier_name': 'Techtronix Components Ltd.'}}], 'trace': [{'step': 1, 'type': 'action', 'thought': 'Need to gather delivery performance data for Techtronix Components Ltd. first', 'tool': 'query_delivery_performance', 'args': {'supplier_name': 'Techtronix Components Ltd.'}, 'latency_ms': 2749, 'result_sources': ['on_time_delivery_report.txt', 'po_with

### Step 2.6 - Observability Trace (Best Practice)

Architect takeaway:
- enterprise agents require transparent step traces
- trace data supports governance, troubleshooting, and stakeholder trust


In [36]:
for row in agent_result["trace"]:
    print(
        f"step={row['step']} | type={row['type']} | tool={row['tool']} | "
        f"latency_ms={row['latency_ms']} | thought={row['thought'][:100]}"
    )


step=1 | type=action | tool=query_delivery_performance | latency_ms=2749 | thought=Need to gather delivery performance data for Techtronix Components Ltd. first
step=2 | type=action | tool=query_quality_incidents | latency_ms=1981 | thought=Need to assess quality incidents before forming recommendation
step=3 | type=action | tool=query_contract_terms | latency_ms=2426 | thought=Need to review contract terms including pricing, obligations, and renewal conditions to complete the
step=4 | type=action | tool=render_result_dashboard | latency_ms=4112 | thought=I have gathered delivery performance (90/100, strong on-time), quality incidents (85/100 with 2 inci
step=5 | type=final | tool=None | latency_ms=6264 | thought=Techtronix shows strong delivery performance (90/100, mostly early/on-time) but has concerning quali


## Summary and Architecture Takeaways

This pro-code solution intentionally separates two architectural modes:
- **Deterministic grounding workflow** for fixed, single-path business questions.
- **Agentic workflow** for non-deterministic, multi-step business decisions.

### 1) Grounding Takeaways

Grounding is the **default enterprise pattern** when the task is clear and the path is fixed.

Architecture pattern:
1. Retrieve evidence from SAP Document Grounding repository.
2. Augment prompt with retrieved context.
3. Generate answer via SAP orchestration runtime.

| Dimension | Assessment |
| --- | --- |
| **When to choose** | Single question, fixed process, low orchestration complexity |
| **Pros** | Simple, transparent, lower operational overhead, easier onboarding |
| **Limits** | Limited adaptive reasoning, weaker fit for multi-criteria recommendations |

SAP BTP value mapping:
- **SAP Document Grounding (Gen AI Hub)**: managed retrieval over indexed enterprise documents.
- **SAP AI Core**: secure model runtime and deployment abstraction.
- **Gen AI Hub SDK**: stable pro-code API for repeatable implementation.
- **BTP Object Store (S3)**: enterprise data source integrated into the grounding pipeline.

### 2) Agentic Takeaways

Agentic design is justified **only** when workflow steps cannot be fully predetermined.

In this workshop, the renewal decision is non-deterministic because the system must:
- decide which evidence domain to query first,
- adapt next tool choice based on intermediate findings,
- synthesize delivery, quality, and contract evidence into one recommendation.

Core architecture decisions used:

| # | Decision | Purpose |
| --- | --- | --- |
| 1 | **Tool specialization** | Separate delivery, quality, and contract tools |
| 2 | **Structured control protocol** | JSON action/final schema for robust orchestration |
| 3 | **Guardrails** | Minimum tool-use threshold and retry on schema violations |
| 4 | **Provenance by design** | Each tool result includes sources and chunk counts |
| 5 | **Observability and governance** | Per-step trace and explicit human approval state |

Key design trade-offs:

| Trade-off | Benefit | Cost |
| --- | --- | --- |
| Agent flexibility vs reliability | Increases decision quality for complex tasks | Requires stricter controls (schema, retries, safeguards) |
| Tool granularity vs maintenance | Finer-grained tools improve controllability | Increases prompt/tool contract maintenance |
| Autonomy vs governance | Higher autonomy accelerates analysis | Business-critical decisions still require HITL checkpoints |
| Lightweight ReAct vs frameworks | Improves teaching transparency | Framework-based orchestration better for production scale |

SAP BTP value mapping for agentic solutions:
- **SAP AI Core**: enterprise runtime boundary, resource-group control, and model endpoint management.
- **Gen AI Hub SDK OrchestrationService**: consistent LLM invocation layer for controller and tools.
- **Document Grounding APIs**: shared retrieval backbone for all tools, preserving evidence consistency.
- **BTP platform services**: identity, security, and operational alignment for production-readiness.

### 3) Tools Architecture Maturity (Workshop to Enterprise)

For this workshop, the presentation capability is implemented as a local tool (`render_result_dashboard`) so participants can clearly learn the ReAct pattern without extra infrastructure.

For production design, a practical maturity path is:
1. **Local tool (workshop/pilot)**: fast learning, low setup, transparent debugging.
2. **Shared internal tool service**: central reusable visualization logic across teams.
3. **Trusted MCP visualization server**: governed enterprise boundary for presentation artifacts and cross-agent reuse.

Best-practice guidance when moving to MCP-backed tools:
- Keep a strict JSON tool contract (schema-validated inputs and outputs).
- Separate concerns: agent decides **what** to present; MCP service decides **how** to render.
- Enforce trust controls: allowlisted server, scoped credentials, and audited calls.
- Minimize data exposure: send summarized decision payloads unless raw evidence is required.
- Implement graceful fallback: if MCP is unavailable, return plain text or local HTML output.

This approach preserves workshop simplicity while teaching a realistic path to enterprise-grade tool architecture.

### Final Architectural Principle

Start with the simplest SAP-native pattern that solves the business problem:
- use **grounding-only** for deterministic Q&A,
- add **agentic orchestration** only when business decisions require adaptive, multi-step evidence reasoning.

This is the practical path from pilot demos to production-grade BTP AI solutions.

## Appendix Pointer

The full architecture appendix has been moved to a dedicated document for the architecture discussion segment:

**[`appendix_architecture.md`](appendix_architecture.md)**

| Appendix | Topic |
|----------|-------|
| **A** | From Notebook Prototype to CAP Application |
| **B** | End-to-End Data Integration from SAP S/4HANA |
| **C** | Connecting Your Agent to Business Applications (Go-Live) |


---

## Cleanup

Run this cell at the end of the workshop to clear sensitive data from memory.

In [37]:
def cleanup_session():
    """
    Clean up sensitive data and reset state at end of workshop.
    
    This clears:
    - Cached OAuth tokens
    - Environment variables from memory
    - Any stored results
    """
    global grounding_client, orchestration_service
    
    print("Cleaning up workshop session...")
    
    # Clear grounding client (includes cached token)
    if grounding_client is not None:
        grounding_client._access_token = None
        grounding_client._token_expires_at = 0.0
        print("   [OK] Cleared cached OAuth token")
    
    # Clear sensitive environment variables from memory
    sensitive_vars = [
        "AICORE_CLIENT_ID",
        "AICORE_CLIENT_SECRET", 
        "AICORE_AUTH_URL",
    ]
    for var in sensitive_vars:
        if var in os.environ:
            del os.environ[var]
    print("   [OK] Cleared sensitive environment variables")
    
    # Clear any agent results that might contain business data
    vars_to_clear = ['agent_result', 'exercise_result', 'baseline_response']
    cleared = []
    for var in vars_to_clear:
        if var in globals():
            globals()[var] = None
            cleared.append(var)
    if cleared:
        print(f"   [OK] Cleared result variables: {cleared}")
    
    print("\nCleanup complete.")
    print("You can safely close this notebook or restart the kernel.")

# Uncomment to run cleanup:
# cleanup_session()